In [2]:
library(tidyverse)
library(edgeR)
library(ggrastr)
library(gridExtra)
library(RColorBrewer)
library(patchwork)
theme_set(theme_bw())

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: limma


Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine




In [1]:
# Differential Accessibility Region (DAR) Analysis with Custom Normalization
#
# This function performs differential accessibility analysis using edgeR,
# incorporating custom normalization based on user-supplied effective library sizes 
# The specific use case is for normalizing ATAC-seq using counts from promoter regions of genes that are invariantly expressed in control and BAF-inhibited samples.
#
# Inputs are:
#     data,  A matrix of raw counts (peaks x samples).
#     coldata, A data frame containing sample metadata. Must include:
#      `Condition`: Factor or character vector indicating experimental groups (e.g., control, treatment).
#      `sizes`: Numeric vector of effective library sizes (i.e., sum of counts at invariant promoters)
#
# This function returns a data frame with differential test results for each region, including logFC, p-values, and FDR
# As well as a classification of each element as gaining ("up"), losing ("down"), or not changing ("nc") in accessibility
# Based on FDR and logFC cutoffs of 0.05 and log2(1.5), respectively

dar <- function(data, coldata) {
    library(edgeR)
  
  # Create DGEList object with counts and group information
    dge <- DGEList(counts = data, group = coldata$Condition)
  
  # Override default library size with effective library sizes from control regions
    dge$samples$lib.size <- coldata$sizes
  
  # Set normalization factors to 1 (i.e., use lib.size as-is with no further scaling)
    dge$samples$norm.factors <- rep(1, nrow(coldata))
  
  # Create design matrix for comparison (model without intercept, group-specific columns)
    design <- model.matrix(~0 + group, data = dge$samples)
    colnames(design) <- levels(dge$samples$group)  # optional: label columns by condition names
  
  # Estimate dispersion 
    dge <- estimateDisp(dge, design)
  
  # Fit generalized linear model (GLM) 
    dge_fit <- glmQLFit(dge, design)
  
  # Conduct differential test (treatment vs control)
    diff <- glmQLFTest(dge_fit, contrast = c(1, -1))
  
  # Return all regions without sorting; include logFC, p-value, FDR, etc.
    res <- topTags(diff, n = Inf, sort.by = "none")$table 
  # Add column categorizing peaks as gaining, losing, or not changing accessibility
    res <- res %>% mutate(
                    change = case_when(FDR < 0.05 & logFC > log2(2) ~ "up",
                                       FDR < 0.05 & logFC < -log2(2) ~ "down",
                                       TRUE ~ "nc"))
  return(res)
}

# This function implements the dar() function separately for TSS-distal and TSS-proximal peaks. 
# In addition to the data and coldata params required by dar(), this function additionall requires
# a 'rowdata' object (should contain minimally genomic coordinates and TSS proximity for each row)

# This function returns a list object where the first element is the table of results from DAR analysis, 
# And the second is a reformatted rowdata object (rows reordered so that they match the results table)

dar_splitTSS <- function(data, coldata, rowdata){ 
    #Use row metadata to split counts matrix into TSS-proximal and dista
    #Run DAR analysis separately on each
    res_tss <- dar(data=data[rowdata$TSS==TRUE,], coldata)
    res_nontss <- dar(data=data[rowdata$TSS==FALSE,], coldata)

    #combine TSS and nonTSS DAR results
    res <- rbind(res_tss, res_nontss) %>% mutate(BAFdep=case_when(change == "down" ~ TRUE, TRUE ~ FALSE)) #add column that binarizes peaks into BAF-dependent (significant accessibility loss) and independent
    #reorder rowdata
    rowdata <- rbind(rowdata[rowdata$TSS==TRUE,], rowdata[rowdata$TSS==FALSE,])
    return(list("results"=res, "row_meta"=rowdata))
}

In [2]:
#read in counts data
countmat <- read.delim("/mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics/inputs/countmat/H1/H1_consensus_filt.tsv", header=F, sep='\t')
#read in row metadata
row_meta <- read.delim("/mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics/inputs/bed/ATAC_peaks/H1/filtered/H1_consensus_filt_annot.tsv", sep='\t')

#read in column metadata
col_meta <- data.frame(Condition=c(rep("C_Control", 3), rep("B_ACBI1", 3), rep("A_A4-KD",3)), rep=rep(1:3,3))

#filter to only include counts (exclude genomic coordinates)
counts <- countmat[,5:13]



In [63]:
#read in control counts (reads over promoters of genes invariantly expressed in BAFinh 6h/10µM vs DMSO
control_mat <- read.delim("/mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics/inputs/countmat/H1/H1_invariant_promoters.tsv", header=F, sep='\t')
#calculate effective library sizes for normalization (sum of counts over each control region set)
lib_sizes <- colSums(control_mat[,5:13])

#add library sizes to column metadata
col_meta <- col_meta %>% mutate(sizes=lib_sizes)

In [64]:
#DAR analysis
dar_ACBI1 <- dar_splitTSS(data=counts[,col_meta$Condition %in% c("C_Control", "B_ACBI1")], coldata=col_meta[col_meta$Condition %in% c("C_Control", "B_ACBI1"),], row_meta) 
dar_A4KD <- dar_splitTSS(data=counts[,col_meta$Condition %in% c("C_Control", "A_A4-KD")], coldata=col_meta[col_meta$Condition %in% c("C_Control", "A_A4-KD"),], row_meta) 

In [68]:
write.table(cbind(dar_ACBI1$row_meta[,1:3],dar_ACBI1$results %>% mutate(BAFdep=case_when(change == "down" ~ TRUE, TRUE ~ FALSE))), "H1_ACBI1_DAR.tsv", quote=FALSE, row.names=FALSE, sep='\t')
write.table(cbind(dar_A4KD$row_meta[,1:3],dar_A4KD$results %>% mutate(BAFdep=case_when(change == "down" ~ TRUE, TRUE ~ FALSE))), "H1_SMARCA4-KD_DAR.tsv", quote=FALSE, row.names=FALSE, sep='\t')

In [69]:
head(dar_ACBI1$results)

,logFC,logCPM,F,PValue,FDR,change,BAFdep
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<lgl>
1,0.22115269,6.843794,9.1672526,0.003054157,0.01123155,nc,FALSE
4,0.05683820,4.390230,0.1163496,0.733662755,0.82031053,nc,FALSE
6,0.07363391,7.351576,1.4146585,0.236783637,0.36629658,nc,FALSE
10,-0.03020429,5.299191,0.0571797,0.811444555,0.87536402,nc,FALSE
11,0.19292781,5.677676,3.0052991,0.085726611,0.16937453,nc,FALSE
12,-0.05247901,7.076201,0.5576994,0.456743310,0.59271777,nc,FALSE


In [3]:
row_meta

seqnames,start,end,width,strand,TSS,H3K4me3,H3K4me1,H3K27ac,H3K27me3,CTCF,type,inDMSO,inACBI1,inACBI1.1,ID
<chr>,<int>,<int>,<int>,<chr>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<chr>,<lgl>,<lgl>,<lgl>,<int>
chr1,778366,779299,934,*,TRUE,TRUE,FALSE,TRUE,FALSE,TRUE,prom,TRUE,TRUE,TRUE,1
chr1,797430,797966,537,*,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,other_nonTSS,TRUE,TRUE,FALSE,2
chr1,804756,805016,261,*,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,other_nonTSS,TRUE,FALSE,FALSE,3
chr1,817333,818369,1037,*,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,other_TSS,TRUE,TRUE,TRUE,4
chr1,818971,819172,202,*,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,other_nonTSS,FALSE,TRUE,FALSE,5
chr1,826738,828129,1392,*,TRUE,TRUE,FALSE,TRUE,FALSE,FALSE,prom,TRUE,TRUE,TRUE,6
chr1,844547,844987,441,*,FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,poised_enh,TRUE,TRUE,TRUE,7
chr1,845693,846001,309,*,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,other_nonTSS,TRUE,TRUE,TRUE,8
chr1,856704,857041,338,*,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,other_nonTSS,TRUE,TRUE,TRUE,9
